# Unity 카탈로그에서 객체 생성 및 구성

## 데모 시나리오: 대학 데이터 플랫폼
**업계 상황: 고등 교육**
이 데모에서는 현대 대학을 위한 포괄적인 데이터 플랫폼을 구축합니다. 대학은 여러 영역에 걸쳐 데이터를 구성하고 관리해야 합니다.
- **학생정보시스템** : 학생 프로필, 재학기록, 학업기록
- **강좌 관리**: 강좌 카탈로그, 수업 일정, 강사 배정  
- **학업 성과**: 학생 성공을 위한 성적, 평가, 분석
- **연구 및 규정 준수**: 교수진 연구 데이터, 규제 보고 요구 사항
Azure Databricks 데이터 엔지니어로서 Unity Catalog가 다음과 같이 어떻게 도움이 되는지 보여드리겠습니다.
- 교육 데이터 자산 전체에 일관된 명명 규칙을 적용합니다.
- 대학 조직 구조에 맞는 카탈로그 및 스키마 생성
- 정형 및 비정형 교육 데이터를 위한 테이블, 뷰, 볼륨 구축
- 교직원 및 관리자를 위한 셀프 서비스 분석을 활성화하는 동시에 거버넌스 제어를 구현합니다.
샘플 교육 데이터를 설정하는 것부터 시작해 보겠습니다.

### 설정: 샘플 교육 데이터 로드
다음 셀을 실행하여 대학 데모용 샘플 테이블을 만듭니다.
- **학생**: 학생 등록 및 인구통계 정보
- **강좌**: 학과 및 강사가 포함된 강좌 카탈로그
- **등록** : 학기별 학생 수강신청 건수
- **평가**: 성적 및 평가 결과

In [0]:
from scripts.setup import setup, setup_03

spark.sql("USE CATALOG trainer_demo")
setup_03(spark)

### 🎯 데모: 샘플 데이터 탐색

In [0]:
%sql
-- Set our working context to the education schema
USE CATALOG trainer_demo;
USE SCHEMA demo_03;

-- Preview student data
SELECT * FROM students LIMIT 5;

In [0]:
%sql
-- Preview course catalog
SELECT * FROM courses LIMIT 5;

In [0]:
%sql
-- Preview enrollment data
SELECT 
  e.enrollment_id,
  s.first_name,
  s.last_name,
  c.course_name,
  e.semester,
  e.year,
  e.enrollment_status
FROM enrollments e
INNER JOIN students s ON e.student_id = s.student_id
INNER JOIN courses c ON e.course_id = c.course_id
LIMIT 10;

## 명명 규칙 적용

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.apply-naming-conventions.png)

### 🎯 데모: 대학 명명 전략
대학 플랫폼의 경우 메달리온 아키텍처에 따라 일관된 명명 전략을 적용합니다.
```txt
university_data (Catalog)
├── bronze (Schema – raw ingestion)
│   ├── raw_student_info_system
│   ├── raw_course_registrations
│   └── raw_learning_management_system
│
├── silver (Schema – cleaned and validated)
│   ├── students
│   ├── courses
│   ├── enrollments
│   └── assessments
│
└── gold (Schema – analytics-ready)
    ├── student_performance_summary
    ├── course_enrollment_trends
    ├── department_analytics
    └── vw_student_success_metrics
```

**주요 명명 규칙:**
- 가독성을 위해 **밑줄이 있는 소문자**를 사용하세요.
- 테이블과 구별하기 위해 `vw_`을 붙인 **접두사 보기**
- 데이터 품질 계층을 위한 **레이어 기반 스키마**(브론즈/실버/골드)
- 사업 목적을 나타내는 **설명적 이름**

#### 🎯 데모: 카탈로그 및 스키마 구성 표시

In [0]:
%sql
-- List all catalogs in the metastore
SHOW CATALOGS;

In [0]:
%sql
-- Show schemas within our training catalog
SHOW SCHEMAS IN trainer_demo;

In [0]:
%sql
-- List all tables in the education schema
SHOW TABLES IN trainer_demo.demo_03;

## 카탈로그 생성

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/create-and-organize-objects-in-unity-catalog/media/3-catalog-definition.png)

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.create-catalog.png)

### 🎯 데모: 대학 카탈로그 생성
이 데모에서는 대학의 데이터 플랫폼에 대한 개발 및 프로덕션 환경을 분리하기 위한 카탈로그를 생성합니다.

 **시나리오**: 대학에는 다음을 위해 별도의 환경이 필요합니다.
- 데이터 엔지니어가 생산 시스템에 영향을 주지 않고 실험할 수 있도록 합니다.
- 생산 과정에서 학생 기록에 대한 엄격한 접근 통제를 유지합니다.
- 환경별 비용 추적 및 리소스 할당 가능

In [0]:
# Determine the storage root
storage_root = (
    spark.sql("DESCRIBE CATALOG EXTENDED adbtraining")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print(f"Storage root: {storage_root}")

In [0]:
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS university_dev
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Development environment for university data engineering'
""")

In [0]:
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS university_prod
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Production environment for university analytics and reporting'
""")

In [0]:
%sql
-- View recently created catalogs
SELECT catalog_name
FROM system.information_schema.catalogs
WHERE catalog_name LIKE 'university%';

In [0]:
%sql
-- Describe catalog to see properties
DESCRIBE CATALOG EXTENDED university_dev;

#### 🎯 데모: 카탈로그 탐색기 사용
**강사 참고 사항**: 카탈로그 탐색기를 시연하려면 UI로 전환하세요.
1. 왼쪽 사이드바에서 **카탈로그**를 클릭합니다.
2. 새로 생성된 `university_dev` 및 `university_prod` 카탈로그를 표시합니다.
3. 계층 구조 설명: 카탈로그 → 스키마 → 테이블
4. 자동으로 생성된 `default` 및 `information_schema` 스키마를 지적합니다.
5. 작업공간 바인딩 및 특정 작업공간에 대한 카탈로그 액세스를 제한하는 방법에 대해 논의합니다.

## 스키마 생성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.create-schema.png)

### 🎯 데모: 메달리온 아키텍처용 스키마 생성
브론즈, 실버, 골드 스키마를 생성하여 처리 단계별로 데이터를 구성합니다.
**시나리오**: 대학은 메달리온 아키텍처를 구현합니다.
- **브론즈**: 소스 시스템(SIS, LMS 등)의 원시 데이터
- **실버**: 정리, 검증 및 중복 제거된 데이터
- **골드**: 집계된 분석 및 보고 데이터 세트

In [0]:
%sql
-- Create bronze schema for raw ingestion
CREATE SCHEMA IF NOT EXISTS university_dev.bronze
COMMENT 'Raw data ingestion layer from university source systems';

In [0]:
%sql
-- Create silver schema for cleaned data
CREATE SCHEMA IF NOT EXISTS university_dev.silver
COMMENT 'Cleaned and validated university data';

In [0]:
%sql
-- Create gold schema for analytics
CREATE SCHEMA IF NOT EXISTS university_dev.gold
COMMENT 'Aggregated analytics and reporting datasets';

In [0]:
%sql
-- List schemas in the development catalog
SHOW SCHEMAS IN university_dev;

In [0]:
%sql
-- Set our working context to the silver schema
USE CATALOG university_dev;
USE SCHEMA silver;

-- Confirm current context
SELECT current_catalog(), current_schema();

## 테이블 및 뷰 생성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.create-tables-views-materialized.png)

### 🎯 데모: 관리되는 테이블 생성
데모_03 스키마에서 데이터를 복사하여 실버 스키마에 관리형 테이블을 생성합니다.

**시나리오**: 적절한 스키마 정의 및 제약 조건을 사용하여 학생 및 강좌 데이터를 실버 레이어로 이동합니다.

In [0]:
%sql
-- Create managed table for students with primary key
CREATE TABLE IF NOT EXISTS university_dev.silver.students (
  student_id INT NOT NULL,
  first_name STRING,
  last_name STRING,
  email STRING,
  enrollment_date DATE,
  program STRING,
  year_level INT,
  gpa DOUBLE,
  CONSTRAINT students_pk PRIMARY KEY (student_id)
)
COMMENT 'Student enrollment and demographic information'
TBLPROPERTIES ('quality' = 'silver', 'domain' = 'student_records');

In [0]:
%sql
-- Populate the table with data from our demo schema
INSERT INTO university_dev.silver.students
SELECT * FROM trainer_demo.demo_03.students;

In [0]:
%sql
-- Create courses table
CREATE TABLE IF NOT EXISTS university_dev.silver.courses (
  course_id STRING NOT NULL,
  course_name STRING,
  department STRING,
  credits INT,
  level STRING,
  instructor STRING,
  CONSTRAINT courses_pk PRIMARY KEY (course_id)
)
COMMENT 'University course catalog';

In [0]:
%sql
-- Populate courses table
INSERT INTO university_dev.silver.courses
SELECT * FROM trainer_demo.demo_03.courses;

In [0]:
%sql
-- Create enrollments table with foreign keys
CREATE TABLE IF NOT EXISTS university_dev.silver.enrollments (
  enrollment_id INT NOT NULL,
  student_id INT,
  course_id STRING,
  semester STRING,
  year INT,
  enrollment_status STRING,
  CONSTRAINT enrollments_pk PRIMARY KEY (enrollment_id),
  CONSTRAINT fk_enrollment_student FOREIGN KEY (student_id) REFERENCES university_dev.silver.students(student_id),
  CONSTRAINT fk_enrollment_course FOREIGN KEY (course_id) REFERENCES university_dev.silver.courses(course_id)
)
COMMENT 'Student course enrollments by semester';

In [0]:
%sql
-- Populate enrollments
INSERT INTO university_dev.silver.enrollments
SELECT * FROM trainer_demo.demo_03.enrollments;

In [0]:
%sql
-- Verify table creation
SHOW TABLES IN university_dev.silver;

### 🎯 데모: 표준 뷰 생성
일반적인 쿼리를 단순화하고 비즈니스 친화적인 인터페이스를 제공하는 보기를 만듭니다.

**시나리오**: 교수진과 관리자는 복잡한 SQL을 작성하지 않고도 등록 통계에 쉽게 액세스할 수 있어야 합니다.

In [0]:
%sql
-- Create view for active enrollments with student and course details
CREATE OR REPLACE VIEW university_dev.silver.vw_active_enrollments AS
SELECT 
  e.enrollment_id,
  e.student_id,
  s.first_name,
  s.last_name,
  s.email,
  s.program,
  c.course_id,
  c.course_name,
  c.department,
  c.instructor,
  e.semester,
  e.year,
  e.enrollment_status
FROM university_dev.silver.enrollments e
INNER JOIN university_dev.silver.students s ON e.student_id = s.student_id
INNER JOIN university_dev.silver.courses c ON e.course_id = c.course_id
WHERE e.enrollment_status = 'Enrolled';

In [0]:
%sql
-- Query the view
SELECT * FROM university_dev.silver.vw_active_enrollments LIMIT 10;

In [0]:
%sql
-- Create view for student performance summary
CREATE OR REPLACE VIEW university_dev.silver.vw_student_gpa_summary AS
SELECT 
  program,
  year_level,
  COUNT(*) as student_count,
  ROUND(AVG(gpa), 2) as avg_gpa,
  ROUND(MIN(gpa), 2) as min_gpa,
  ROUND(MAX(gpa), 2) as max_gpa
FROM university_dev.silver.students
GROUP BY program, year_level
ORDER BY program, year_level;

In [0]:
%sql
-- Query the GPA summary view
SELECT * FROM university_dev.silver.vw_student_gpa_summary;

### 🎯 데모: 골드 레이어 집합 생성
사전 집계된 비즈니스 측정항목을 사용하여 골드 레이어 테이블을 만듭니다.
**시나리오**: 대학의 리더십 대시보드에는 매일 새로 고쳐지는 등록 지표가 필요합니다.

In [0]:
%sql
-- Create gold table for department enrollment summary
CREATE OR REPLACE TABLE university_dev.gold.department_enrollment_summary AS
SELECT 
  c.department,
  e.semester,
  e.year,
  COUNT(DISTINCT e.student_id) as unique_students,
  COUNT(e.enrollment_id) as total_enrollments,
  COUNT(DISTINCT e.course_id) as courses_offered
FROM university_dev.silver.enrollments e
INNER JOIN university_dev.silver.courses c ON e.course_id = c.course_id
WHERE e.enrollment_status IN ('Enrolled', 'Completed')
GROUP BY c.department, e.semester, e.year;

In [0]:
%sql
-- Query the gold aggregation
SELECT * FROM university_dev.gold.department_enrollment_summary
ORDER BY year DESC, semester, department;

### 🎯 데모: 외부 테이블
**강사 참고 사항**: 외부 테이블은 Unity Catalog의 관리형 스토리지 외부에 저장된 데이터를 쿼리해야 할 때 유용합니다. 대학 환경에서는 다음과 같습니다.
- 과거 데이터는 여전히 데이터 레이크 컨테이너에 저장되어 있습니다.
- 연구 협력의 공유 데이터 세트
- 규정 준수를 위해 특정 저장 위치에 보관되어야 하는 보관된 기록

**예시 시나리오**: 대학은 원래 위치에 유지되어야 하지만 Unity 카탈로그를 통해 쿼리할 수 있어야 하는 졸업 기록을 ADLS Gen2에 보관했습니다.

```sql
-- Example: Create external table (requires pre-configured external location)
CREATE EXTERNAL TABLE IF NOT EXISTS university_dev.bronze.archived_graduates
LOCATION 'abfss://archives@universitydata.dfs.core.windows.net/graduates/'
COMMENT 'Historical graduation records in external storage';
```

*강사 참고 사항: 먼저 Unity 카탈로그에서 외부 위치를 구성해야 합니다.*

## 볼륨 생성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.create-volumes.png)

### 🎯 데모: 관리 볼륨 생성
강의 계획서, 강좌 자료, 학생 문서 등 표 형식이 아닌 파일에 대한 관리 볼륨을 생성합니다.

**시나리오**: 대학에는 다음을 위한 관리형 스토리지가 필요합니다.
- 강좌 강의 계획서(PDF 파일)
- 과제 제출(다양한 형식)
- 학생 성공 예측을 위한 머신러닝 모델
- 강좌용 이미지 및 멀티미디어 콘텐츠

In [0]:
%sql
-- Create volume for course materials
CREATE VOLUME IF NOT EXISTS university_dev.silver.course_materials
COMMENT 'Storage for course syllabi, assignments, and learning resources';

In [0]:
%sql
-- Create volume for ML models
CREATE VOLUME IF NOT EXISTS university_dev.gold.ml_models
COMMENT 'Trained machine learning models for student analytics';

In [0]:
%sql
-- List volumes in the silver schema
SHOW VOLUMES IN university_dev.silver;

#### 🎯 데모: 볼륨의 파일 작업
Python을 사용하여 볼륨 내 파일 작업을 시연합니다.

In [0]:
# Get the volume path
volume_path = "/Volumes/university_dev/silver/course_materials"

# List files in the volume (will be empty initially)
files = dbutils.fs.ls(volume_path)
print(f"Files in course_materials volume: {len(files)}")
for file in files:
    print(f"  - {file.name}")

In [0]:
# Create a sample syllabus file
sample_syllabus = """
COURSE SYLLABUS
Course: CS101 - Introduction to Programming
Instructor: Dr. Sarah Mitchell
Semester: Fall 2024

Course Description:
This course introduces fundamental programming concepts using Python.

Grading:
- Assignments: 40%
- Midterm: 25%
- Final Project: 35%
"""

# Write the file to the volume
syllabus_path = f"{volume_path}/CS101_syllabus.txt"
dbutils.fs.put(syllabus_path, sample_syllabus, overwrite=True)

print(f"Created sample syllabus at: {syllabus_path}")

In [0]:
# Read the file back
content = dbutils.fs.head(syllabus_path)
print("Syllabus content:")
print(content)

In [0]:
# Demonstrate using volumes with pandas
import pandas as pd

# Create a sample dataframe with student feedback
feedback_data = {
    'course_id': ['CS101', 'CS201', 'DE101', 'BA101'],
    'semester': ['Fall 2024', 'Fall 2024', 'Spring 2024', 'Fall 2024'],
    'avg_rating': [4.5, 4.2, 4.8, 4.3],
    'response_count': [45, 38, 52, 41]
}

df = pd.DataFrame(feedback_data)

# Save as CSV in the volume
csv_path = f"{volume_path}/course_feedback.csv"
df.to_csv(csv_path, index=False)

print("Saved course feedback CSV to volume")
print(df)

In [0]:
%sql
-- Query CSV directly using SQL
SELECT * FROM csv.`/Volumes/university_dev/silver/course_materials/course_feedback.csv`;

### 🎯 데모: 외부 볼륨
**강사 참고 사항**: 외부 볼륨은 데이터를 이동하지 않고도 기존 클라우드 스토리지 위치에 대한 거버넌스를 제공합니다.
**예시 시나리오**: 대학의 LMS(학습 관리 시스템)는 학생 제출물을 Azure Blob Storage 컨테이너로 내보냅니다. 파일을 이동하지 않고 Unity Catalog 거버넌스를 추가하고 싶습니다.
```sql
-- Example: Create external volume (requires pre-configured external location)
CREATE EXTERNAL VOLUME IF NOT EXISTS university_dev.bronze.lms_exports
LOCATION 'abfss://lms-data@universitydata.dfs.core.windows.net/exports/'
COMMENT 'LMS system exports - externally managed storage';
```

*강사 참고 사항: 먼저 적절한 자격 증명이 구성된 외부 위치가 필요합니다.*

## DDL 작업 구현

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.implement-ddl-operations.png)

### 🎯 데모: 재사용 가능한 함수 만들기
대학별 비즈니스 로직을 캡슐화하는 SQL 함수를 만듭니다.
**시나리오**: 대학에는 다음에 대한 표준화된 계산이 필요합니다.
- GPA 문자 등급 변환
- 학위과정별 학점요건
- 학업적 지위 판정

In [0]:
%sql
-- Create function to convert numeric GPA to letter grade
CREATE OR REPLACE FUNCTION university_dev.silver.gpa_to_letter_grade(gpa DOUBLE)
RETURNS STRING
COMMENT 'Convert numeric GPA (0-4.0 scale) to letter grade'
RETURN CASE
  WHEN gpa >= 3.7 THEN 'A'
  WHEN gpa >= 3.3 THEN 'A-'
  WHEN gpa >= 3.0 THEN 'B+'
  WHEN gpa >= 2.7 THEN 'B'
  WHEN gpa >= 2.3 THEN 'B-'
  WHEN gpa >= 2.0 THEN 'C+'
  WHEN gpa >= 1.7 THEN 'C'
  WHEN gpa >= 1.0 THEN 'D'
  ELSE 'F'
END;

In [0]:
%sql
-- Test the function
SELECT 
  student_id,
  first_name,
  last_name,
  gpa,
  university_dev.silver.gpa_to_letter_grade(gpa) as letter_grade
FROM university_dev.silver.students
LIMIT 10;

In [0]:
%sql
-- Create function to determine academic standing
CREATE OR REPLACE FUNCTION university_dev.silver.academic_standing(gpa DOUBLE)
RETURNS STRING
COMMENT 'Determine academic standing based on GPA'
RETURN CASE
  WHEN gpa >= 3.5 THEN 'Dean\'s List'
  WHEN gpa >= 3.0 THEN 'Good Standing'
  WHEN gpa >= 2.0 THEN 'Satisfactory'
  WHEN gpa >= 1.5 THEN 'Academic Warning'
  ELSE 'Academic Probation'
END;

In [0]:
%sql
-- Use the function in a query
SELECT 
  academic_standing(gpa) as standing,
  COUNT(*) as student_count,
  ROUND(AVG(gpa), 2) as avg_gpa
FROM university_dev.silver.students
GROUP BY academic_standing(gpa)
ORDER BY avg_gpa DESC;

#### 🎯 데모: Python 함수
텍스트 정규화와 같은 더 복잡한 논리를 위한 Python 함수를 만듭니다.

In [0]:
%sql
-- Create Python function to standardize email domains
CREATE OR REPLACE FUNCTION university_dev.silver.extract_email_domain(email STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Extract domain from email address'
AS $$
  if email and '@' in email:
    return email.split('@')[1].lower()
  return None
$$;

In [0]:
%sql
-- Use the Python function
SELECT 
  extract_email_domain(email) as email_domain,
  COUNT(*) as student_count
FROM university_dev.silver.students
GROUP BY extract_email_domain(email);

### 🎯 데모: ALTER TABLE 작업
테이블 구조를 수정하여 새 열을 추가하고 제약 조건을 업데이트합니다.


**시나리오**: 대학에서는 유학생 상태를 추적하고 졸업 날짜 추적을 추가하려고 합니다.

In [0]:
%sql
-- Add a new column to the students table
ALTER TABLE university_dev.silver.students
ADD COLUMN is_international BOOLEAN
COMMENT 'Indicates if student is international';

In [0]:
%sql
-- Add another column for expected graduation
ALTER TABLE university_dev.silver.students
ADD COLUMN expected_graduation_date DATE
COMMENT 'Expected graduation date based on program requirements';

In [0]:
%sql
-- Update the new columns with sample data
UPDATE university_dev.silver.students
SET is_international = (student_id % 5 = 0),
    expected_graduation_date = DATE_ADD(enrollment_date, (4 - year_level) * 365 + 365);

In [0]:
%sql
-- Verify the changes
SELECT student_id, first_name, last_name, year_level, is_international, expected_graduation_date
FROM university_dev.silver.students
LIMIT 10;

In [0]:
%sql
-- Describe the table to see the updated schema
DESCRIBE TABLE EXTENDED university_dev.silver.students;

### 🎯 데모: ALTER CATALOG 작업
카탈로그 속성 및 소유권을 수정합니다.

**시나리오**: 비용 추적을 위해 카탈로그에 태그를 지정하고 생산을 위한 예측 최적화를 활성화합니다.

In [0]:
%sql
-- Add tags to catalog for governance
ALTER CATALOG university_dev SET TAGS (
  'environment' = 'development',
  'cost_center' = 'academic_affairs',
  'data_classification' = 'internal'
);  

In [0]:
%sql
-- View catalog details to see applied tags
DESCRIBE CATALOG EXTENDED university_dev;

**강사 참고 사항**: 추가 ALTER 작업 시연:
- 스키마 속성을 수정하려면 `ALTER SCHEMA`
- 열 이름을 바꾸려면 `ALTER TABLE ... RENAME COLUMN`(Delta Lake 열 매핑 필요)
- 열을 제거하려면 `ALTER TABLE ... DROP COLUMN`
- CDC의 `delta.enableChangeDataFeed`과 같은 테이블 속성

## 해외 카탈로그 구현

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/create-and-organize-objects-in-unity-catalog/media/8-foreign-catalogs-reason.png)

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.implement-foreign-catalog.png)

### 🎯 데모: 외국 카탈로그 개념
**강사 참고 사항**: 외부 카탈로그를 사용하면 데이터 마이그레이션 없이 Unity 카탈로그를 통해 직접 외부 데이터베이스를 쿼리할 수 있습니다.
**대학 시나리오**: 
대학의 기존 학생 정보 시스템(SIS)은 PostgreSQL에서 실행됩니다. 테라바이트급의 기록 데이터를 마이그레이션하는 대신 외부 카탈로그를 생성하여 Unity 카탈로그 거버넌스를 유지하면서 직접 쿼리할 수 있습니다.

**대학이 제공하는 혜택**:
1. **통합 거버넌스**: PostgreSQL에 대한 모든 쿼리는 Unity Catalog의 액세스 제어를 거칩니다.
2. **감사 추적**: 레거시 데이터에 대한 모든 액세스는 Unity 카탈로그에 기록됩니다.
3. **데이터 중복 없음**: 레거시 데이터가 PostgreSQL에 유지되어 동기화 복잡성을 방지합니다.
4. **증분 마이그레이션**: Databricks로 점진적으로 마이그레이션하면서 레거시 데이터에 액세스하세요.
**구현 단계**(데모용):
```sql
-- Step 1: Create connection to external PostgreSQL database
CREATE CONNECTION university_sis_db TYPE POSTGRESQL
OPTIONS (
  host 'sis-database.university.edu',
  port '5432',
  user secret ('university-secrets', 'postgres-user'),
  password secret ('university-secrets', 'postgres-password')
);

-- Step 2: Create foreign catalog using the connection
CREATE FOREIGN CATALOG IF NOT EXISTS legacy_sis
USING CONNECTION university_sis_db
OPTIONS (database 'student_information_system')
COMMENT 'Legacy Student Information System on PostgreSQL';

-- Step 3: Query external data through Unity Catalog
SELECT * FROM legacy_sis.public.historical_transcripts
WHERE graduation_year >= 2010;

-- Step 4: Join foreign catalog data with Unity Catalog tables
SELECT 
  s.student_id,
  s.first_name,
  s.last_name,
  h.degree_program as previous_degree,
  h.graduation_year as previous_grad_year
FROM university_dev.silver.students s
INNER JOIN legacy_sis.public.alumni_history h 
  ON s.email = h.email
WHERE h.graduation_year >= 2015;
```

**지원되는 외부 시스템**:
- 포스트그레SQL
- MySQL
- SQL Server / Azure SQL 데이터베이스
- 눈송이
- 빅쿼리
- 적색편이
*참고: 이 데모에는 외부 데이터베이스 연결과 적절한 네트워크 구성이 필요합니다.*

## AI/BI Genie 지침 구성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.configure-ai-bi-genie-instructions.png)

### 🎯 데모: AI/BI 지니 구성
**강사 참고 사항**: AI/BI Genie를 사용하면 비즈니스 사용자가 자연어를 사용하여 데이터를 쿼리할 수 있습니다. 적절한 구성은 Genie가 대학 관련 용어를 이해하는 데 도움이 됩니다.
**대학 시나리오**:
지도교수와 학과장은 등록 동향을 분석해야 하지만 SQL에 익숙하지 않습니다. AI/BI Genie를 사용하면 "학과별 등록 동향을 보여주세요"와 같은 질문을 하고 정확한 결과를 얻을 수 있습니다.

#### 구성 단계(UI 데모)
**1. 지니 스페이스 만들기**
- Databricks 작업 영역에서 AI/BI로 이동합니다.
- "University Enrollment Analytics"라는 새로운 Genie 공간을 만듭니다.
- 테이블 추가: `students`, `courses`, `enrollments`, `vw_active_enrollments`(university_dev.silver 스키마에서)

**2. 지식 저장소 구성**

**테이블 메타데이터 개선 사항**:```
students table description:
"Contains student enrollment records including demographics, 
academic program, GPA, and enrollment dates. Use this table 
when analyzing student population or academic performance."

Key column synonyms:
- program: "major", "degree program", "field of study"
- year_level: "class year", "grade level", "academic year"
- gpa: "grade point average", "academic performance score"
```

**3. 관계 정의**:```
enrollments.student_id = students.student_id (Many to One)
enrollments.course_id = courses.course_id (Many to One)
```

**4. SQL 지침 추가**:
```sql
-- Example Query 1: "Show enrollment trends by department"
SELECT 
  c.department,
  e.semester,
  e.year,
  COUNT(DISTINCT e.student_id) as student_count
FROM university_dev.silver.enrollments e
JOIN university_dev.silver.courses c ON e.course_id = c.course_id
WHERE e.enrollment_status = 'Enrolled'
GROUP BY c.department, e.semester, e.year
ORDER BY e.year DESC, e.semester, c.department;

-- Example Query 2: "Which students are on the Dean's List?"
SELECT 
  student_id,
  first_name,
  last_name,
  program,
  gpa
FROM university_dev.silver.students
WHERE gpa >= 3.5
ORDER BY gpa DESC;

-- Example Query 3: "Show courses with the highest enrollment"
SELECT 
  c.course_name,
  c.department,
  c.instructor,
  COUNT(e.enrollment_id) as enrollment_count
FROM university_dev.silver.courses c
JOIN university_dev.silver.enrollments e ON c.course_id = e.course_id
WHERE e.enrollment_status = 'Enrolled'
GROUP BY c.course_name, c.department, c.instructor
ORDER BY enrollment_count DESC
LIMIT 10;
```

**5. SQL 표현식 정의**:```
Name: Dean's List Students
Expression: students.gpa >= 3.5
Synonyms: "high achievers", "top students", "honors students"

Name: Active Enrollment
Expression: enrollments.enrollment_status IN ('Enrolled', 'In Progress')
Synonyms: "current enrollment", "ongoing courses"
```

**6. 자연어 쿼리 테스트**:
- "각 프로그램의 학생 수는 몇 명입니까?"
- "Sarah Mitchell 박사가 가르치는 컴퓨터 공학 과정을 보여주세요"
- "이번 가을에 가장 많은 등록이 이루어진 학과는 어디입니까?"
- "데이터 공학과의 Dean's List 학생은 누구인가요?"

#### 데모 혜택
**교수 및 고문용**:
- SQL 지식이 필요하지 않습니다.
- 등록 및 학업 성과에 대한 셀프 서비스 분석
- 임시 질문에 대한 빠른 답변

**데이터 엔지니어용**:
- 중앙집중화된 지식관리
- 임시 쿼리 요청 감소
- 쿼리 전반에 걸쳐 일관된 비즈니스 로직

**강사 노트**: 반복적인 개선 프로세스를 보여줍니다.
1. 실패하는 질문을 하라
2. 생성된 SQL을 검토합니다.
3. 수정 후 명령어로 추가
4. 문제를 다시 테스트하여 개선 사항을 보여줍니다.

---

## 데모 요약
이 데모에서는 대학 데이터 플랫폼의 관점을 통해 Unity 카탈로그에서 객체를 생성하고 구성하는 전체 라이프사이클을 다루었습니다.
✅ **메달리온 아키텍처에 따른 명명 규칙 적용**  
✅ 환경 격리를 위한 **카탈로그 생성**(개발/프로덕션)  
✅ 처리 레이어(브론즈/실버/골드)별로 데이터를 구성하는 **스키마 생성**  
✅ 적절한 제약 조건과 관계를 갖춘 **빌드된 테이블과 뷰**  
✅ 표 형식이 아닌 대학 파일을 위한 **생성된 볼륨**  
✅ 함수, ALTER 문을 포함한 **구현된 DDL 작업**  
✅ 레거시 시스템 통합을 위한 **해외 카탈로그 탐색**  
✅ 자연어 분석을 위해 **구성된 AI/BI Genie**
**Azure Databricks 데이터 엔지니어를 위한 주요 사항**:
1. 조직의 구조에 맞는 **명확한 명명 전략으로 시작**
2. **격리를 위해 카탈로그 사용**(환경, 민감도, 규정 준수)
3. **스키마를 활용하여 도메인 또는 처리 단계별로 데이터 구성**
4. 시행되지 않더라도 **제약조건 정의**(PK/FK) - 쿼리 최적화에 도움이 됩니다.
5. **재사용 가능한 함수 생성**을 통해 비즈니스 로직 표준화
6. 구조화된 테이블과 함께 구조화되지 않은 데이터를 관리하려면 **볼륨을 사용**하세요.
7. 비즈니스 사용자를 위한 셀프 서비스 분석을 활성화하려면 **Genie 공간을 구성**하세요.
여기에 설명된 원칙은 산업 전반에 걸쳐 적용됩니다. 특정 비즈니스 도메인 및 거버넌스 요구 사항에 맞게 이름과 조직을 조정하세요.